<div style="background: linear-gradient(125deg, #080b12 0%, #151b2c 58%, #4b102d 100%); border: 1px solid #343b52; border-radius: 18px; padding: 30px 34px; margin: 4px 0 26px; color: #ffffff; box-shadow: 0 12px 30px rgba(0,0,0,0.22);">
  <div style="font-size: 14px; font-weight: 700; letter-spacing: 0.16em; color: #ff5a5f; text-transform: uppercase;">AMD × vLLM Semantic Router Workshop</div>
  <div style="font-size: 38px; line-height: 1.15; font-weight: 800; margin-top: 10px;">Route One Agent Across Two Models</div>
  <div style="font-size: 18px; color: #d7dbea; margin-top: 8px;">Build one stable application endpoint that chooses the right model for each request.</div>
  <div style="display: flex; flex-wrap: wrap; align-items: stretch; gap: 10px; margin-top: 24px;">
    <div style="flex: 1 1 145px; background: rgba(255,255,255,0.08); border-radius: 10px; padding: 13px 16px;"><div style="font-size: 11px; letter-spacing: 0.12em; color: #aeb6cc;">APPLICATION</div><div style="font-size: 18px; font-weight: 700; margin-top: 4px;">One agent</div></div>
    <div style="display: flex; align-items: center; color: #ff5a5f; font-size: 24px; font-weight: 700;">→</div>
    <div style="flex: 1 1 190px; background: rgba(255,90,95,0.14); border: 1px solid rgba(255,90,95,0.4); border-radius: 10px; padding: 13px 16px;"><div style="font-size: 11px; letter-spacing: 0.12em; color: #ffadb0;">STABLE ENDPOINT</div><div style="font-size: 18px; font-weight: 700; margin-top: 4px;">vllm-sr/auto</div></div>
    <div style="display: flex; align-items: center; color: #72d6ff; font-size: 24px; font-weight: 700;">→</div>
    <div style="flex: 1 1 190px; background: rgba(114,214,255,0.10); border-radius: 10px; padding: 13px 16px;"><div style="font-size: 11px; letter-spacing: 0.12em; color: #9ee4ff;">MODEL PATHS</div><div style="font-size: 16px; font-weight: 700; margin-top: 4px;">Routine model<br>Reasoning model</div></div>
  </div>
  <div style="margin-top: 18px; font-size: 13px; color: #aeb6cc;">AMD Instinct GPU · vLLM · Semantic Router · Hermes Agent</div>
</div>

**One agent. One endpoint. Two model paths.**

Most applications begin with one model endpoint. That is simple, but every
request pays for the same model, even when some requests are quick and others
need deeper reasoning.

In this workshop, you will put one stable endpoint in front of two models and
teach it when to use each one. You will try the route in a browser, call it
through an API, connect an agent, inspect why a decision happened, and create a
policy for your own application.

The path we will build is:

```text
application need → observable signal → routing decision → selected model
```


## 1. Two Model Endpoints on Azure Foundry

The first model handles direct, interactive work. The second is reserved for
requests that benefit from deeper reasoning.

| Model path | Model | Intended work |
| --- | --- | --- |
| Routine model | `FW-Qwen3.8-27B` | Definitions, summaries, and direct instructions |
| Reasoning model | `FW-GLM-5.3-Flash` | Diagnosis, comparison, planning, and multi-step analysis |

Both models are served from Azure Foundry. This VM reaches them through a local
relay on port 8002 (routine) and port 8001 (reasoning), so the rest of the
workshop talks to them like two local OpenAI-compatible servers. There is
nothing to start by hand.


### Start vLLM Semantic Router and Verify Services

Load the workshop helper. The first code cell starts the Foundry relay, the Router with d3-lite, and the Dashboard, then reports the status
of the complete workshop environment.


In [ ]:
import importlib
import workshop_lab

importlib.reload(workshop_lab)
lab = workshop_lab.lab
lab.welcome()
services = lab.start_platform()
services = lab.status()


In [ ]:
models = lab.show_models()

if models["routine"] and models["reasoning"]:
    print("Both model endpoints are ready.")
else:
    print("A model endpoint is not reachable yet; wait a moment, then run this cell again.")


Send one request directly to each model. This establishes a useful
baseline: both backends work before the router enters the path.


In [ ]:
if models["routine"]:
    routine_backend_result = lab.direct_model_check("routine")
else:
    routine_backend_result = None
    print("Skipped until the routine model is reachable.")


In [ ]:
if models["reasoning"]:
    reasoning_backend_result = lab.direct_model_check("reasoning")
else:
    reasoning_backend_result = None
    print("Skipped until the reasoning model is reachable.")


## 2. Set Up vLLM Semantic Router with Two Models

An application should not have to choose a model before every request. We will
give it one OpenAI-compatible endpoint:

```text
application → Semantic Router (vllm-sr/auto) → selected model on Azure Foundry
                     │
                     ├── asks d3-lite (decision model, this VM's CPU)
                     └── decision evidence → Dashboard
```

The setup cell in Section 1 already started a prebuilt two-model routing policy
so you can experience the system before studying its YAML. In Section 7, you
will rebuild this configuration one part at a time.

`lab.start_platform()` runs `/opt/workshop/bin/start-platform.sh`. The script
loads `/opt/workshop/config/router.yaml`, starts the Foundry relay, the Router
(which serves `vllm-sr/auto` on port 8898 and runs d3-lite), and the Dashboard,
and waits until they are ready.


## 3. vLLM Semantic Router Dashboard Demo

Open the Dashboard and log in with the username (`u`) and password (`p`) printed
below the button. Then go to **Playground** and select `vllm-sr/auto`.


In [ ]:
lab.browser_links()


Start with a request that should be routine:

> Give me a one-sentence definition of HTTP.

Before you send it, predict the model. After it completes, record the selected
decision, model, and latency.

Now try a request designed to match our reasoning policy:

> Diagnose cascading failures in a distributed queue, compare multiple
> remediation strategies and their tradeoffs, and justify a safe rollout plan.

Predict again and send it. Then select **Outcomes → Insights**.
Under **Insight Records**, find the two newest requests in the **Routing
Insights** table. Open each record to see the evidence used by the router,
not just the model’s answer.


### See the Complexity Policy the Router Is Using

The prebuilt policy comes from:

```text
/opt/workshop/config/router.yaml
```

When the platform starts, that file is copied to the active configuration:

```text
/workspace/generated-config/router.yaml
```

Two parts of it decide the route:

- the **decision model** in `global.model_catalog`: d3-lite, a small model
  from the vLLM Semantic Router project that runs on this VM's CPU; and
- the **complexity signal** in `recipes[0].signals.complexity`, which asks the
  decision model how difficult each request is.

Run the next cell to read the actual active configuration rather than a
separate example.


In [ ]:
from pathlib import Path
import yaml

active_config_path = Path("/workspace/generated-config/router.yaml")
active_config = yaml.safe_load(
    active_config_path.read_text(encoding="utf-8")
)
active_recipe = active_config["recipes"][0]
reasoning_rules = next(
    decision["rules"]
    for decision in active_recipe["decisions"]
    if decision["name"] == "reasoning-lane"
)

print(f"Read from: {active_config_path}")
print()
print(yaml.safe_dump(
    {
        "model_catalog": active_config["global"]["model_catalog"],
        "complexity": active_recipe["signals"]["complexity"],
        "reasoning-lane rules": reasoning_rules,
    },
    sort_keys=False,
))


### How the Decision Model Becomes a Route

For every request, the Router asks d3-lite one question: is this request
**easy**, **medium**, or **hard**? d3-lite answers with a probability for each
level, and the signal reports the most likely one:

```text
easy 0.04   medium 0.64   hard 0.32   →   request_complexity: medium
```

There are no example lists or thresholds to maintain: d3-lite was trained to
judge how much reasoning a request needs.

The active reasoning decision contains this rule:

```yaml
operator: OR
conditions:
  - type: complexity
    name: request_complexity
    label: hard
  - type: complexity
    name: request_complexity
    label: medium
```

A signal asks a question about the request. A condition refers to one named
answer. Here a `hard` **or** `medium` answer makes the reasoning lane eligible;
`easy` requests continue to the routine fallback.


## 4. Test Routing Through the Application API

This is a prediction exercise with a scorecard.

The Dashboard showed that routing works. Now call the same OpenAI-compatible
endpoint from Python and compare the observed route with your prediction.

Keep the known routine prompt as a control, then replace the three placeholders
below:

1. a request you expect the routine model to handle;
2. a request you expect the reasoning model to handle; and
3. a request whose route you genuinely cannot predict.

Write down each prediction first. Every result shows the measured complexity,
winning decision, and selected model.


In [ ]:
routing_experiment = [
    {
        "label": "control",
        "prompt": "Give me a short definition of HTTP.",
        "prediction": "routine",
    },
    {
        "label": "my clearly routine prompt",
        "prompt": "YOUR ROUTINE PROMPT HERE",
        "prediction": "routine",
    },
    {
        "label": "my clearly hard prompt",
        "prompt": "YOUR HARD PROMPT HERE",
        "prediction": "reasoning",
    },
    {
        "label": "my uncertain prompt",
        "prompt": "YOUR AMBIGUOUS PROMPT HERE",
        "prediction": "uncertain",
    },
]

routing_experiment


In [ ]:
if any("YOUR " in item["prompt"] for item in routing_experiment):
    print("Replace the three YOUR ... placeholders, predict the routes, then rerun.")
    routing_results = []
else:
    routing_results = lab.compare_predictions(routing_experiment)


For each result, compare:

- `prediction`: what you expected;
- `observed_complexity`: the level d3-lite reported;
- `observed_probabilities`: how sure it was about each level;
- `observed_decision`: which policy matched; and
- `observed_model`: where the request was sent.

If you see `medium`, d3-lite judged the request moderately difficult; in this
policy that is enough for the reasoning model. Keep such prompts. They show
where the boundary sits when you tune the policy.

### Improve the Router from What You Observed

Use the result that surprised you most as calibration evidence:

- If too many requests reach the reasoning model, route only `hard` there and
  let `medium` fall back to the routine model.
- If d3-lite's idea of difficulty does not match your application, ask it your
  own question with a `systemone` signal, for example *"Does answering need
  multi-step reasoning about our billing rules?"*
- If complexity is correct but the route is wrong for your application, add
  another condition instead of trying to force complexity to represent a
  different fact.

If complexity is not enough, vLLM Semantic Router provides other
[signal types](https://vllm-sr.ai/docs/tutorials/signal/overview) for
application-specific facts:

| Application fact | Useful signal type |
| --- | --- |
| A question only your application can phrase | `systemone` |
| An exact incident or business marker appears | `keyword` |
| The request is too large for one model | `context` |
| The application labels the workload | `metadata` |
| The request contains personal information | `pii` |
| The request includes an image or audio | `input_modality` |
| The conversation is multi-turn or tool-heavy | `conversation` |

A practical policy may combine them:

```text
hard AND public                 -> reasoning model
PII OR confidential marker      -> local model
hard AND NOT private            -> cloud-eligible model
```

Use `AND` when every requirement must hold, `OR` when any trigger should use
the same route, and `NOT` for an explicit exclusion.

There is no target number of conditions. Start with the smallest rule that
expresses the requirement clearly, then add a condition only when the route
needs another independent fact. In Section 7 you will build the complexity
signal and connect it to a decision. In Section 8 you will add a second signal
and observe how priority resolves two matching decisions.


## 5. Connect Hermes Agent to vLLM Semantic Router

This section checks one application integration:

```text
Can Hermes use vllm-sr/auto, edit a project, and finish with passing tests?
```

Hermes knows only the model name `vllm-sr/auto`. It does not know which
backend model serves the request.

Before running it, open the visible `agent-demo/` folder beside this
notebook. `pricing.py` contains the duplicated logic, `reporting.py` uses the
public functions, and `test_pricing.py` defines the behavior that must remain
unchanged.

This run succeeds when Hermes completes the edit and the tests pass. Section 6
demonstrates model changes across several user turns.


In [ ]:
AGENT_TASK = (
    "Read this small project, identify the duplicated pricing logic, extract "
    "it into one private helper without changing public behavior, and run "
    "python -m pytest -q."
)

exercise_dir = lab.prepare_agent_demo()
print("Task Hermes will receive:")
print(AGENT_TASK)


The setup cell copied the visible `agent-demo/` project into a clean working
directory. Hermes edits the copy, so the original files remain available for
comparison. Run the next cell and wait for:

```text
Hermes final response
3 passed
```

The progress timer confirms that the agent is still running. You do not need
to watch the Dashboard during this step.


In [ ]:
agent_run = lab.run_agent(AGENT_TASK, exercise_dir)


### Verify the Hermes Agent Workflow

The final response describes the change, and `3 passed` verifies the edited
project.

```text
Hermes → vllm-sr/auto → selected backend → file edit → passing tests
```

The integration stays the same as the routing policy grows. Hermes continues
to call `vllm-sr/auto`; conditions for context length, conversation shape,
private data, or application metadata belong in the Router configuration,
not in the agent.

Inspecting internal model calls is optional.

To inspect them, select **Outcomes → Insights** and look under
**Insight Records → Routing Insights**.


## 6. vLLM Semantic Router Multi-Turn Simulation

Now test routing behavior directly, without the agent harness in the middle.
The next cell sends four user turns through one conversation:

1. define a technical term;
2. rewrite one sentence;
3. compare several coding strategies; and
4. summarize the recommendation.

Predict the model for each turn before running the cell. Every result prints
the latest user message, complexity class, decision, and selected model.


In [ ]:
conversation = []
turn_results = []

turns = [
    "Define idempotency in one sentence.",
    "Rewrite this sentence clearly: retries can make the same job run twice.",
    (
        "Compare multiple refactoring strategies for the duplicated pricing logic, "
        "explain their tradeoffs, and justify a safe implementation plan."
    ),
    "Summarize your recommendation in one sentence.",
]

for prompt in turns:
    conversation, observed = lab.routed_turn(conversation, prompt)
    turn_results.append(observed)


Read `turn_results` from top to bottom.

The comparison request should be the clearest reasoning candidate. The direct
requests should be rated `easy` and use the routine path.

If a result differs from your prediction, keep that prompt for the calibration
exercise. The mismatch tells you whether the policy should route a different
set of labels or ask d3-lite a more specific question.


## 7. Build the vLLM Semantic Router Configuration

You have used a prebuilt route from the Dashboard, Python, and Hermes. Now
rebuild the same routing pattern from the smallest useful configuration.

To configure vLLM Semantic Router, you create a YAML policy. The repository
contains
many complete [configuration examples](https://github.com/vllm-project/semantic-router/tree/main/config),
but this section builds the smallest useful policy one layer at a time.

The basic customization loop is:

```text
define a signal
→ reference that signal from a decision
→ assign eligible models
→ test the observed route
```

The [signal guide](https://vllm-sr.ai/docs/tutorials/signal/overview)
describes the available observations. Adding a signal definition alone does
not change routing. A new signal takes effect only after a decision references
it.
Decision rules combine their conditions with `AND`, `OR`, or `NOT`, as shown
in the [decision configuration guide](https://vllm-sr.ai/docs/tutorials/decision/overview#configuration).

| Stage | What you build | Question it answers |
| --- | --- | --- |
| 1 | One model and a fallback decision | Where does unmatched traffic go? |
| 2 | A second model path | Which backends are available? |
| 3 | A decision model and a complexity signal | What did the Router observe? |
| 4 | A reasoning decision | What should happen when the signal matches? |

Every stage writes a complete YAML file under `generated-config/` beside the
notebook. The important configuration areas are:

| Configuration area | Purpose |
| --- | --- |
| `listeners` | Defines where application requests enter |
| `providers.models` | Connects logical model names to backend endpoints and context sizes |
| `global.model_catalog` | Names the decision model the Router asks (d3-lite) |
| `recipes[].signals` | Extracts evidence from each request |
| `recipes[].decisions` | Converts evidence into an eligible model path |
| `priority` | Resolves cases where several decisions match |
| `algorithm` | Selects among models assigned to a decision |
| `entrypoints` | Exposes the stable application-facing name `vllm-sr/auto` |

Focus on the highlighted fields at each stage. `lab.validate_config()` checks
each file with the Router and boots it with the bundled runtime. From Stage 4
the boot also loads d3-lite, so that check takes about half a minute.


### Stage 1: Define the Routine Model and Fallback

```text
application → vllm-sr/auto → routine-model
```

Pay attention to these fields:

- `listeners`: the routed API address, port, and request timeout;
- `providers.models`: the logical model name and its backend endpoint (the local Foundry relay);
- `provider_model_id`: the Foundry deployment name the backend serves;
- `context_window_size`: the model's context capacity;
- `recipes`: a named routing policy, here `default`;
- `entrypoints`: the stable name applications request, mapped to that recipe; and
- `routine-lane`: the fallback decision for every unmatched request.


In [ ]:
from pathlib import Path
import yaml

NOTEBOOK_DIR = Path(workshop_lab.__file__).resolve().parent
CONFIG_DIR = NOTEBOOK_DIR / "generated-config"
CONFIG_DIR.mkdir(exist_ok=True)

stage1_yaml = '''
version: v0.5

listeners:
  - name: workshop
    address: 127.0.0.1
    port: 8898
    timeout: 600s

providers:
  models:
    - name: routine-model
      api_format: openai
      provider_model_id: FW-Qwen3.8-27B
      backend_refs:
        - name: routine-backend
          endpoint: 127.0.0.1:8002
          protocol: http
          weight: 1
      context_window_size: 65536

recipes:
  - name: default
    signals: {}
    decisions:
      - name: routine-lane
        description: Send every request to the routine model.
        priority: 100
        rules:
          operator: AND
          conditions: []
        modelRefs:
          - model: routine-model
        algorithm:
          type: static

entrypoints:
  - model_names: [vllm-sr/auto]
    recipe: default
'''

stage1_path = CONFIG_DIR / "01-one-model.yaml"
stage1_path.write_text(stage1_yaml.strip() + "\n", encoding="utf-8")

print("Saved the complete one-model configuration to:")
print(stage1_path)
print()
print(stage1_path.read_text(encoding="utf-8"))

lab.validate_config(stage1_path)


### What Stage 1 Establishes

The fallback contains:

```yaml
operator: AND
conditions: []
```

An `AND` rule succeeds when every condition succeeds. With no conditions,
there is nothing to reject, so this decision matches every request.

Also notice:

- `modelRefs` assigns `routine-model` to this decision;
- `priority: 100` gives later, more specific rules room to override it; and
- `algorithm.type: static` directly selects the assigned model.

The generated file is an artifact, not the active configuration. In Section 8
you will deploy a policy into the running Router.


### Stage 2: Add the Reasoning Model

```text
                                  ┌──► routine-model
application → vllm-sr/auto ───────┤
                                  └──► reasoning-model
```

Pay attention to the new provider entry:

- `name`: the logical name decisions will refer to, `reasoning-model`;
- `backend_refs`: how to reach it (the relay's reasoning port);
- `provider_model_id`: must match the Foundry deployment name; and
- `context_window_size`: must not exceed the backend's actual capacity.

This stage connects the model but does not make it eligible for any request.


In [ ]:
stage2 = yaml.safe_load(stage1_path.read_text(encoding="utf-8"))

reasoning_provider = {
    "name": "reasoning-model",
    "api_format": "openai",
    "provider_model_id": "FW-GLM-5.3-Flash",
    "backend_refs": [
        {
            "name": "reasoning-backend",
            "endpoint": "127.0.0.1:8001",
            "protocol": "http",
            "weight": 1,
        }
    ],
    "context_window_size": 65536,
}

stage2["providers"]["models"].append(reasoning_provider)

stage2_path = CONFIG_DIR / "02-two-models.yaml"
stage2_path.write_text(
    yaml.safe_dump(stage2, sort_keys=False),
    encoding="utf-8",
)

print("Added this provider model:")
print(yaml.safe_dump(reasoning_provider, sort_keys=False))
print("Saved the complete configuration to:")
print(stage2_path)
lab.validate_config(stage2_path)


### What Stage 2 Changes

Both easy and difficult prompts still follow the fallback.

The new model is reachable, but no decision refers to it:

```text
connected model ≠ selectable route
```

A decision must make `reasoning-model` eligible for a request.


### Stage 3: Add the Decision Model and the Complexity Signal

The router needs evidence before it can make a semantic choice. A **signal**
extracts that evidence. The complexity signal gets it by asking a **decision
model**, so this stage adds both.

Pay attention to:

- `global.model_catalog.deployments.d3-lite`: the model files (`artifact`),
  and that it runs on the CPU in `bfloat16`;
- `system.decision_model`: the deployment every model-backed signal asks;
- the signal's `name`: the stable identifier decisions will reference; and
- its `description`: what this signal is for, in your own words.

The signal reports one of:

```text
request_complexity: easy
request_complexity: medium
request_complexity: hard
```


In [ ]:
stage3 = yaml.safe_load(stage2_path.read_text(encoding="utf-8"))

decision_model = {
    "deployments": {
        "d3-lite": {
            "provider": "model_runtime",
            "artifact": "/opt/models/d3-lite",
            "device": "cpu",
            "dtype": "bfloat16",
            "public_name": "d3-lite",
        }
    },
    "system": {"decision_model": {"deployment": "d3-lite"}},
}
complexity_signal = {
    "name": "request_complexity",
    "description": "Separate direct requests from reasoning-heavy work.",
}

stage3["global"] = {"model_catalog": decision_model}
stage3["recipes"][0]["signals"]["complexity"] = [complexity_signal]

stage3_path = CONFIG_DIR / "03-add-signal.yaml"
stage3_path.write_text(
    yaml.safe_dump(stage3, sort_keys=False),
    encoding="utf-8",
)

print("Added the decision model and this signal:")
print(yaml.safe_dump(
    {"model_catalog": decision_model, "complexity": [complexity_signal]},
    sort_keys=False,
))
print("Saved the complete configuration to:")
print(stage3_path)
lab.validate_config(stage3_path)


The new signal can measure complexity, but requests still follow the fallback
because no decision uses that measurement yet.

The tuning choice is no longer a threshold. It is **which answers** send a
request to the reasoning model:

```text
hard only           → fewer, clearly difficult requests escalate
medium or hard      → anything beyond a direct answer escalates
```

Tune in this order:

1. collect prompts that represent your application;
2. label the route you want for each prompt;
3. choose the labels that send work to the reasoning model; and
4. if d3-lite's notion of difficulty is not the fact you need, add a
   `systemone` signal that asks your own question.

Under-routing may reduce answer quality. Over-routing costs more latency and
more tokens on the larger model. Your application determines which error costs
more.

After the final policy is deployed, send labelled prompts from **Playground**
and open their records under **Outcomes → Insights**. The record's signal
values show d3-lite's easy, medium, and hard probabilities.


### Stage 4: Route Medium and Hard Prompts to the Reasoning Model

The policy is now:

```text
if request_complexity is medium or hard:
    use reasoning-model
otherwise:
    use routine-model
```

Pay attention to:

- `conditions.type: complexity`, which selects the signal family;
- `conditions.name: request_complexity`, which selects your signal;
- `conditions.label`, which selects one of its answers;
- `operator: OR`, so either answer is enough;
- `priority: 200`, which beats the fallback's priority of 100;
- `modelRefs`, which assigns `reasoning-model`; and
- `algorithm.type: static`, which selects that assigned model directly.


In [ ]:
stage4 = yaml.safe_load(stage3_path.read_text(encoding="utf-8"))

reasoning_decision = {
    "name": "reasoning-lane",
    "description": "Use the stronger model for medium and hard work.",
    "priority": 200,
    "rules": {
        "operator": "OR",
        "conditions": [
            {"type": "complexity", "name": "request_complexity", "label": "hard"},
            {"type": "complexity", "name": "request_complexity", "label": "medium"},
        ],
    },
    "modelRefs": [
        {
            "model": "reasoning-model",
            "use_reasoning": True,
        }
    ],
    "algorithm": {"type": "static"},
}

stage4["recipes"][0]["decisions"].insert(0, reasoning_decision)

stage4_path = CONFIG_DIR / "04-add-decision.yaml"
stage4_path.write_text(
    yaml.safe_dump(stage4, sort_keys=False),
    encoding="utf-8",
)

print("Added this higher-priority decision:")
print(yaml.safe_dump(reasoning_decision, sort_keys=False))
print("Saved the complete configuration to:")
print(stage4_path)
lab.validate_config(stage4_path)


Trace one hard request through the completed policy:

```text
1. request_complexity reports hard (or medium)
2. reasoning-lane matches
3. routine-lane also matches because it is the fallback
4. priority 200 beats priority 100
5. the static algorithm selects reasoning-model
```

Now each configuration concept names something visible in that trace:

| Concept | What you can observe |
| --- | --- |
| Signal | `request_complexity:hard` |
| Decision | `reasoning-lane` matched |
| Priority | `200` won over fallback `100` |
| Algorithm | `static` selected the assigned path |
| Model | `reasoning-model` served the request |


## 8. Add an Application-Specific Routing Rule

Imagine an incident-response assistant. Difficult diagnosis normally belongs
on the reasoning model, but a request marked as an active production incident
must receive a fast acknowledgement first.

Write the application requirement before writing YAML:

> When an active-incident marker appears, use the routine model immediately,
> even if the request also looks reasoning-heavy.

The new configuration adds:

```text
keyword signal  active-incident
decision        incident-fast-lane
priority        300, higher than reasoning-lane at 200
modelRefs       routine-model
```

The matching `signal.name` and `condition.name` connect the observed keyword to
the decision. The next cell writes one participant-facing file:

```text
generated-config/05-incident-policy.yaml
```

It contains only the routing policy, the recipe's signals and decisions, so
deploying it cannot overwrite listeners, providers, the decision model, or
service settings. The complete configuration used for validation is kept in a
hidden internal directory.

Notice that two different operators appear in this example:

- `operator: OR` inside the keyword signal means any listed phrase can make
  that one signal match.
- `operator: AND` inside the decision combines named signal conditions. With
  one condition it simply requires `active-incident` to match.

Signal operators combine observations inside one detector. Decision operators
combine the named facts used to decide whether a route is eligible.


### Deploy the Updated Policy

1. Run the next cell. It writes `05-incident-policy.yaml`, validates the
   complete configuration, and deploys the policy with `lab.deploy_policy()`.
   The Router reloads it without a restart.
2. Open **Routing → Builder** in the Dashboard and confirm that
   `incident-fast-lane` has priority 300 and selects `routine-model`.
3. Return to the notebook and run the collision test.

`lab.deploy_policy()` replaces the `default` recipe through the Router
management API. It refuses the update if the recipe changed after it was read,
for example from the Dashboard; run the cell again in that case.


In [ ]:
stage5 = yaml.safe_load(stage4_path.read_text(encoding="utf-8"))
recipe = stage5["recipes"][0]

incident_signal = {
    "name": "active-incident",
    "operator": "OR",
    "keywords": [
        "SEV-1",
        "production outage",
        "service unavailable",
    ],
    "case_sensitive": False,
}
incident_decision = {
    "name": "incident-fast-lane",
    "description": "Use the low-latency model during an active incident.",
    "priority": 300,
    "rules": {
        "operator": "AND",
        "conditions": [
            {
                "type": "keyword",
                "name": "active-incident",
            }
        ],
    },
    "modelRefs": [{"model": "routine-model"}],
    "algorithm": {"type": "static"},
}

recipe["signals"]["keywords"] = [incident_signal]
recipe["decisions"].insert(0, incident_decision)

validation_dir = CONFIG_DIR / ".validation"
validation_dir.mkdir(exist_ok=True)
stage5_validation_path = validation_dir / "05-incident-policy.yaml"
stage5_validation_path.write_text(
    yaml.safe_dump(stage5, sort_keys=False),
    encoding="utf-8",
)
stage5_path = CONFIG_DIR / "05-incident-policy.yaml"
stage5_path.write_text(
    yaml.safe_dump({"recipes": [recipe]}, sort_keys=False),
    encoding="utf-8",
)

print("Added this signal:")
print(yaml.safe_dump(incident_signal, sort_keys=False))
print("Added this decision:")
print(yaml.safe_dump(incident_decision, sort_keys=False))
lab.validate_config(stage5_validation_path)
print("Saved the routing policy to:")
print(stage5_path)
print("Next: apply it via the vLLM-SR Dashboard: Routing → Builder → Import, paste the file, then Compile and Deploy.")


### Verify Priority When Both Rules Match

Consider:

> SEV-1 production outage. Diagnose the root cause, compare multiple
> remediation strategies, and recommend a rollout.

This request contains the incident keyword and asks for difficult analysis.
Both `incident-fast-lane` and `reasoning-lane` can match.

The previous cell deployed `05-incident-policy.yaml`. Predict the winner and
run the next cell.
The verification checks that priority 300 selects `incident-fast-lane` and
`routine-model`.


In [ ]:
COLLISION_PROMPT = (
    "SEV-1 production outage. Diagnose the root cause, compare three "
    "remediation plans, and recommend a rollout."
)

if not lab.decision_is_active("incident-fast-lane"):
    print("The incident policy is not active yet.")
    print("Run the previous cell; it deploys 05-incident-policy.yaml.")
    print("Then rerun this cell.")
    print("Active decisions:", lab.active_decisions())
    incident_result = None
else:
    incident_result = lab.routed_chat(COLLISION_PROMPT, max_tokens=400)
    assert incident_result["decision"] == "incident-fast-lane"
    assert incident_result["selected_model"] == "routine-model"
    print(
        "✓ Collision resolved by priority: "
        "incident-fast-lane → routine-model"
    )


## 9. Introduce Select-Mode Mixture of Models

You now have:

```text
one stable application-facing model name
+ separately deployed models with different strengths
+ a policy that chooses how to fulfill each request
= Select-mode Mixture of Models
```

This is different from Mixture of Experts. MoE routes among components inside
one model; MoM coordinates separate model deployments.

Today you built **Select** mode: one request selects one model. A follow-up
workshop can build on the same entrypoint with:

| Mode | Behavior |
| --- | --- |
| Cascade | Start with one model and escalate when needed |
| Fusion | Ask multiple models and synthesize their outputs |
| Workflow | Assign planner, worker, and verifier roles |


## 10. Challenge: Keep Private Finance Requests Local

Imagine that `vllm-sr/auto` belongs to a financial assistant. For this
exercise:

| Model | Application role |
| --- | --- |
| `routine-model` | Smaller model inside the local data boundary |
| `reasoning-model` | Larger model treated as an external service |

In this workshop both models run on Azure Foundry. The different roles
simulate a real deployment boundary.

Your goal is to preserve useful routing for public requests while ensuring
that the private financial prompts in the scorecard stay on the smaller local
model.

### Hints

Build the policy the same way as in Section 8: load the active policy, add a
signal and a decision (or change one), write it to a file, and deploy it with
`lab.deploy_policy()`. You can also edit it in **Routing → Builder** on the
Dashboard, then Compile and Deploy.

1. Add a signal that recognises the private requests.
2. Either edit an existing decision or add a new one.
3. Choose the rule shape that expresses your intent:

| Desired behavior | Rule shape |
| --- | --- |
| Invoke the decision when the signal matches | Add the signal condition to the decision's `AND` group |
| Exclude the decision when the signal matches | Add a `NOT` group to the `AND` group, with the signal condition inside it |

4. A condition names the signal's type (for example `keyword`) and the exact
   name of your signal.
5. Infer the signal details and the appropriate decision from the four test
   prompts.

Deploy your policy, then run the scorecard without changing its expected
results. To claim the workshop prize, show the presenter a passing test cell.


In [ ]:
challenge_prompts = {
    "public_simple": "Define operating margin in one sentence.",
    "public_hard": (
        "Compare multiple strategies for reducing a working-capital "
        "shortfall, explain their tradeoffs, and justify a safe rollout plan."
    ),
    "private_simple": (
        "PRIVATE ACCOUNT-ID 4821: summarize this payment note in one "
        "sentence: invoice is 12 days overdue."
    ),
    "private_hard": (
        "PRIVATE ACCOUNT-ID 4821: compare multiple restructuring "
        "strategies, explain their tradeoffs, and justify a safe rollout plan."
    ),
}

for name, prompt in challenge_prompts.items():
    print(f"{name}: {prompt}")


Before running the scorecard, predict which model should handle each request.

| Request | Required model behavior |
| --- | --- |
| Public simple | Smaller local model |
| Public hard | Larger reasoning model |
| Private simple | Smaller local model |
| Private hard | Smaller local model |

More than one valid Router policy can satisfy this requirement. The scorecard
checks behavior rather than requiring one specific signal name, decision name,
or rule-tree design.


In [ ]:
CHALLENGE_EXPECTED_MODELS = {
    "public_simple": "routine-model",
    "public_hard": "reasoning-model",
    "private_simple": "routine-model",
    "private_hard": "routine-model",
}

challenge_results = {
    name: lab.routed_chat(prompt, max_tokens=400)
    for name, prompt in challenge_prompts.items()
}

failures = []
for name, expected_model in CHALLENGE_EXPECTED_MODELS.items():
    observed = challenge_results[name]
    actual_model = observed["selected_model"]
    marker = "✓" if actual_model == expected_model else "✗"
    print(
        f"{marker} {name}: {observed['decision']} → {actual_model} "
        f"(expected model {expected_model})"
    )
    if actual_model != expected_model:
        failures.append(
            f"{name}: observed {actual_model}, expected {expected_model}"
        )

assert not failures, "Challenge checks failed:\n" + "\n".join(failures)
print()
print("Challenge complete: 4/4 routing checks passed")


You have finished the challenge only when the notebook prints:

```text
Challenge complete: 4/4 routing checks passed
```

Show the passing cell to the presenter to claim the workshop prize. Then open
the four records under **Outcomes → Insights** and explain why each request
selected its model.

A production privacy policy needs stronger controls than the simple markers
used in this exercise. This challenge demonstrates the routing structure that
the future
financial-data workshop will extend with PII detection and data-boundary
verification.


## Workshop Summary

You began with two independent model endpoints. You now have one stable model
name and a policy that can choose between them.

You can:

- connect model paths without changing the application-facing endpoint;
- define evidence with signals;
- turn that evidence into decisions;
- resolve overlapping rules with priority;
- inspect the reason for an individual route;
- connect Hermes Agent through the same endpoint; and
- test a routing policy against prompts from your own workload.

The next workshop can keep this interface and move from selecting one model to
coordinating several.
